<style>
.jp-Notebook, .jp-RenderedHTMLCommon, .rendered_html, .rendered_html table, .rendered_html th, .rendered_html td { font-family: "Microsoft YaHei", "Microsoft JhengHei", "SimHei", "SimSun", "Noto Sans CJK SC", "PingFang SC", Arial, sans-serif !important; }
.rendered_html table { font-size: 13px; }
.rendered_html td { vertical-align: top; }
</style>
# 10人persona问卷作答：有/无背景信息影响分析

本 notebook 对同一批随机抽取的10位CFPS persona进行比较：一版不提供竞品/产品背景信息，一版提供整理后的 Markdown 背景信息。

## 1. 结论导读

- 本 notebook 不嵌入静态SVG；图表由后续代码单元运行生成。
- 重点不再只是运行时间，而是比较背景信息如何改变具体题目答案、哪些persona更容易变化，以及变化集中在哪些问卷模块。
- 本版答案导出已按问卷格式拆分A6/A6a/A6b、A14/A15、C4/C5，并用`NA`标记不适用题，适用但未回答保持空白。
- `comparison_top10_changed_answer_details.csv` 保存了Top变化题目的逐人答案对照和人口学信息。

## 2. 答案导出格式说明

- `A6`为非西医减重措施1-11的尝试次数序列，`A6a`为对应单次平均坚持时长，`A6b`为对应月均花费；没有经历某项措施的位置填0。
- `A14`和`A15`分别代表首诊科室与接受减重治疗科室。
- `C4`和`C5`分别代表6个商品名的易读性与好记程度评分序列。
- `NA`表示根据show_if逻辑不适用；空白表示该题适用但模型未给出答案或答案无法解析。

## 3. C部分产品信息输入示卡

| 产品 | 作用机制 | 治疗疾病布局 | 使用方式 | 减重疗效 | 代谢获益 | 安全性 |
| --- | --- | --- | --- | --- | --- | --- |
| 产品X | GLP-1R/GCGR 双靶点激动剂 | 长期体重管理；MASH（纤维化F2-F4）；不含2型糖尿病适应症 | 皮下注射，每周一次；0.3/0.6/1.2/2.4/3.6/4.8mg 阶梯式剂量爬坡 | 全球数据：76周体重降幅13.4%；中国数据暂无 | 腰围下降16.6cm（46周）；糖化血红蛋白下降1.68%（16周）；收缩压下降9.2mmHg（46周）；舒张压下降4.7mmHg（46周） | 任何不良反应发生率91%；常见不良反应：恶心56%、呕吐27%、腹泻22% |
| 产品A | GLP-1R/GIP 双靶点激动剂 | 长期体重管理；MASH（纤维化F2-F3）；2型糖尿病 | 皮下注射，每周一次；2.5/5/7.5/10/12.5/15mg 阶梯式剂量爬坡 | 全球数据：72周体重降幅17.8%；中国数据：52周体重降幅15.1% | 腰围下降14.5cm（52周）；糖化血红蛋白下降2.07%（72周）；收缩压下降9mmHg（52周）；舒张压下降5.5mmHg（52周） | 任何不良反应发生率90%；常见不良反应：恶心56%、呕吐27%、腹泻22% |
| 产品B | GLP-1R 激动剂 | 长期体重管理；MASH（纤维化F2-F3）；2型糖尿病 | 皮下注射，每周一次；0.25/0.5/1/1.7/2.4mg 阶梯式剂量爬坡 | 全球数据：68周体重降幅12.4%；中国数据：44周体重降幅9.9% | 腰围下降13.5cm（68周）；糖化血红蛋白下降1.6%（68周）；收缩压下降6.6mmHg（68周）；舒张压下降2.8mmHg（68周） | 任何不良反应发生率90%；常见不良反应：恶心56%、呕吐27%、腹泻22% |
| 产品C | GLP-1R/GCGR 双靶点激动剂 | 长期体重管理；MASH（纤维化F2-F3）；2型糖尿病 | 皮下注射，每周一次；2/4/6mg 阶梯式剂量爬坡 | 全球数据暂无；中国数据：48周体重降幅14.3% | 腰围下降10.7cm（48周）；糖化血红蛋白下降1.73%（28周）；收缩压下降8.6mmHg（48周）；舒张压下降5.3mmHg（48周） | 任何不良反应发生率97%；常见不良反应：恶心51%、呕吐43%、腹泻39% |

## 4. 运行状态

| variant_label | status | 人数 |
| --- | --- | --- |
| 不含背景信息 | ok | 10 |
| 含背景信息 | ok | 10 |

## 5. 10位persona人口学与健康画像

| 样本编号 | 年龄 | 性别 | 省份 | 城乡属性 | 学历 | BMI | BMI配额段 | 目标人群细分 | 合并症类别汇总 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 47 | 32 | 女 | 吉林省 | 城镇 | 大学本科 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 |
| 51 | 49 | 女 | 河南省 | 城镇 | 高中/中专/技校/职高 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 |
| 86 | 61 | 女 | 上海市 | 城镇 | 高中/中专/技校/职高 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 |
| 120 | 55 | 男 | 河南省 | 城镇 | 大学本科 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 |
| 123 | 45 | 男 | 湖北省 | 城镇 | 大学本科 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 |
| 138 | 34 | 男 | 山东省 | 城镇 | 大学本科 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 |
| 150 | 32 | 男 | 黑龙江省 | 城镇 | 大学本科 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 |
| 151 | 52 | 男 | 广西壮族自治区 | 城镇 | 高中/中专/技校/职高 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 |
| 187 | 47 | 男 | 河南省 | 城镇 | 大学本科 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 |
| 207 | 55 | 男 | 河南省 | 城镇 | 高中/中专/技校/职高 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 |

## 6. 每位persona受背景影响的题目数量

| pid | 样本编号 | 年龄 | 性别 | 省份 | 城乡属性 | 学历 | BMI | BMI配额段 | 目标人群细分 | 合并症类别汇总 | 变化题目数 | Persona生成线索 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 411244104 | 51 | 49 | 女 | 河南省 | 城镇 | 高中/中专/技校/职高 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 44 | 49岁女，河南省城镇，BMI 30.85（肥胖），高中/中专/技校/职高，收入近似为1.7万-5万，合并症线索：无可识别研究相关合并症，自评健康一般，规律锻炼。 |
| 673100701 | 138 | 34 | 男 | 山东省 | 城镇 | 大学本科 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 42 | 34岁男，山东省城镇，BMI 24.84（超重），大学本科，收入近似为1.7万-5万，合并症线索：关节运动系统，自评健康比较健康，低频锻炼。 |
| 220322103 | 47 | 32 | 女 | 吉林省 | 城镇 | 大学本科 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | 39 | 32岁女，吉林省城镇，BMI 28.04（肥胖），大学本科，收入近似为1.7万-5万，合并症线索：关节运动系统，自评健康很健康，从不锻炼。 |
| 450250102 | 151 | 52 | 男 | 广西壮族自治区 | 城镇 | 高中/中专/技校/职高 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 37 | 52岁男，广西壮族自治区城镇，BMI 26.35（超重），高中/中专/技校/职高，收入近似为1.7万-5万，合并症线索：关节运动系统，自评健康很健康，高频锻炼。 |
| 411875101 | 187 | 47 | 男 | 河南省 | 城镇 | 大学本科 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 37 | 47岁男，河南省城镇，BMI 26.73（超重），大学本科，收入近似为5万-10万，合并症线索：高血压，自评健康比较健康，规律锻炼。 |
| 420493101 | 123 | 45 | 男 | 湖北省 | 城镇 | 大学本科 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | 36 | 45岁男，湖北省城镇，BMI 24.51（超重），大学本科，收入近似为10万以上，合并症线索：高血压；心脑血管，自评健康比较健康，规律锻炼。 |
| 310855102 | 86 | 61 | 女 | 上海市 | 城镇 | 高中/中专/技校/职高 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 32 | 61岁女，上海市城镇，BMI 33.30（肥胖），高中/中专/技校/职高，收入近似为1.7万-5万，合并症线索：无可识别研究相关合并症，自评健康比较健康，从不锻炼。 |
| 411879101 | 120 | 55 | 男 | 河南省 | 城镇 | 大学本科 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 31 | 55岁男，河南省城镇，BMI 24.22（超重），大学本科，收入近似为5万-10万，合并症线索：高血压，自评健康比较健康，规律锻炼。 |
| 211769103 | 150 | 32 | 男 | 黑龙江省 | 城镇 | 大学本科 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | 31 | 32岁男，黑龙江省城镇，BMI 27.13（超重），大学本科，收入近似为1.7万-5万，合并症线索：肝病或MASH线索，自评健康比较健康，从不锻炼。 |
| 411313102 | 207 | 55 | 男 | 河南省 | 城镇 | 高中/中专/技校/职高 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | 26 | 55岁男，河南省城镇，BMI 28.55（肥胖），高中/中专/技校/职高，收入近似为5万-10万，合并症线索：糖尿病，自评健康比较健康，高频锻炼。 |

## 7. 分人群变化概览

| 性别 | BMI配额段 | 目标人群细分 | 人数 | 平均变化题目数 | 最高变化题目数 |
| --- | --- | --- | --- | --- | --- |
| 女 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 3 | 38.3 | 44 |
| 男 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 1 | 26.0 | 26 |
| 男 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 6 | 35.7 | 42 |

## 8. 运行耗时与token

| 版本 | 人数 | 失败人数 | 画像平均秒 | 问卷平均秒 | 画像总tokens | 问卷总tokens | 总tokens |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 不含背景信息 | 10 | 0 | 10.717 | 34.834 | 30578 | 269175 | 299753 |
| 含背景信息 | 10 | 0 | 12.151 | 38.042 | 48758 | 287684 | 336442 |

## 9. Top变化问题总览

| 题目ID | 问卷部分 | 题型 | 答案变化人数 | 答案变化率 | 主要答案迁移 | 题目文本 |
| --- | --- | --- | --- | --- | --- | --- |
| A6 | A 减重旅程 | matrix_numeric | 10 | 1.0 | 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 0 -> 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 1（1人）；0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 0 -> 0, 3, 2, 0, 0, 0, 0, 0, 5, 0, 0（1人）；0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 0 -> 0, 3, 2, 2, 0, 0, 0, 0, 4, 2, 0（1人） | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6b | A 减重旅程 | matrix_numeric | 10 | 1.0 | 0, 200, 0, 300, 0, 0, 0, 0, 0, 100, 0 -> 0, 200, 100, 0, 0, 0, 0, 0, 0, 0, 0（1人）；0, 200, 0, 300, 0, 0, 0, 0, 0, 200, 0 -> 0, 200, 0, 300, 0, 0, 0, 0, 0, 200, 400（1人）；0, 200, 100, 150, 0, 0, 0, 0, 0, 200, 0 -> 0, 200, 150, 200, 0, 0, 0, 0, 0, 0, 400（1人） | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| B16 | B GLP-1药物选择与使用情况 | matrix | 10 | 1.0 | NA -> {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重变化的功能"}}（1人）；NA -> {"A":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}}（1人）；{"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针和复诊提醒、记录体重和血糖变化"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} -> {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针和复诊提醒、记录体重变化"}}（1人） | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B19 | B GLP-1药物选择与使用情况 | matrix_rating_1_7 | 10 | 1.0 | NA -> {"A":{"0":5,"1":5,"2":6,"3":5,"4":3,"5":4,"6":4,"7":4,"8":4,"9":5,"10":5,"11":4,"12":4,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":3,"20":2}}（1人）；NA -> {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":5,"12":5,"13":6,"14":5,"15":6,"16":7,"17":4,"18":5,"19":4,"20":3}}（1人）；{"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":4,"12":4,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} -> {"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":4,"7":4,"8":5,"9":5,"10":6,"11":4,"12":4,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}}（1人） | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B3 | B GLP-1药物选择与使用情况 | matrix_rating_1_7 | 10 | 1.0 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":5,"7":6,"8":4,"9":5,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":5,"20":4} -> {"1":6,"2":7,"3":5,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":5,"20":4}（1人）；{"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5} -> {"1":6,"2":7,"3":6,"4":6,"5":5,"6":5,"7":6,"8":4,"9":5,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5}（1人）；{"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":5,"16":6,"17":4,"18":5,"19":5,"20":4} -> {"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5}（1人） | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| C1 | C 新产品测试 | matrix_rating_1_7 | 10 | 1.0 | {"X":{"1":4,"2":5,"3":5,"4":5,"5":3,"6":4},"A":{"1":6,"2":5,"3":7,"4":6,"5":4,"6":6},"B":{"1":6,"2":5,"3":4,"4":5,"5":4,"6":5},"C":{"1":5,"2":5,"3":5,"4":5,"5":3,"6":4}} -> {"X":{"1":5,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":6,"2":5,"3":7,"4":7,"5":4,"6":6},"B":{"1":6,"2":5,"3":4,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":5,"4":5,"5":3,"6":5}}（1人）；{"X":{"1":4,"2":5,"3":5,"4":5,"5":3,"6":4},"A":{"1":6,"2":5,"3":7,"4":6,"5":4,"6":6},"B":{"1":6,"2":5,"3":5,"4":5,"5":4,"6":5},"C":{"1":5,"2":6,"3":5,"4":5,"5":3,"6":5}} -> {"X":{"1":5,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":7,"2":5,"3":7,"4":7,"5":4,"6":7},"B":{"1":6,"2":5,"3":4,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":5,"4":5,"5":3,"6":5}}（1人）；{"X":{"1":4,"2":5,"3":5,"4":6,"5":3,"6":4},"A":{"1":6,"2":5,"3":7,"4":7,"5":4,"6":6},"B":{"1":6,"2":5,"3":5,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":6,"4":5,"5":3,"6":5}} -> {"X":{"1":4,"2":5,"3":5,"4":5,"5":3,"6":4},"A":{"1":6,"2":5,"3":7,"4":6,"5":4,"6":6},"B":{"1":6,"2":5,"3":5,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":6,"4":5,"5":3,"6":5}}（1人） | 阅读产品X、产品A、产品B、产品C信息后，对各药物在各维度上的表现满意度和总体满意度分别打分。 |
| S10 | 甄别问卷 | multi | 10 | 1.0 | [2,3,4,5,9,10] -> [2,3,4,9,10,13]（1人）；[2,3,4,9,10,12,13] -> [2,3,4,9,11,12,13]（1人）；[2,3,4,9,11,12,13] -> [2,4,9,11,12,13]（1人） | 请问您既往经历过以下哪些减重方式？ |
| A6a | A 减重旅程 | matrix_numeric | 9 | 0.9 | 0, 3, 0, 2, 0, 0, 0, 0, 2, 0, 3 -> 0, 4, 0, 2, 0, 0, 0, 0, 3, 0, 2（1人）；0, 3, 1, 0, 0, 0, 0, 0, 2, 0, 0 -> 0, 3, 0, 2, 0, 0, 0, 0, 2.5, 0, 0（1人）；0, 3, 1, 2, 0, 0, 0, 0, 2, 0, 3 -> 0, 6, 0, 3, 0, 0, 0, 0, 4, 0, 3（1人） | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| B13 | B GLP-1药物选择与使用情况 | matrix_multi | 9 | 0.9 | NA -> {"A":[1,2,6,15]}（1人）；NA -> {"A":[1,2,7]}（1人）；{"A":[1,2,4,7,8,15,16]} -> {"A":[1,2,4,7,8,15]}（1人） | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B14 | B GLP-1药物选择与使用情况 | matrix_titration | 9 | 0.9 | NA -> {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":12,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":20}}（1人）；NA -> {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":20,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":28}}（1人）；{"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":16}} -> {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":16}}（1人） | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| C5 | C 商品名测试 | matrix_rating_1_7 | 9 | 0.9 | 5, 4, 5, 4, 4, 5 -> 5, 4, 5, 6, 4, 5（1人）；5, 4, 5, 4, 4, 5 -> 5, 6, 5, 4, 4, 6（1人）；5, 4, 5, 4, 4, 5 -> 6, 5, 4, 5, 4, 6（1人） | C5 商品名好记程度，按商品名1-6顺序导出。 |
| A8 | A 减重旅程 | matrix_rating_1_7 | 8 | 0.8 | {"1":4,"2":3,"3":3,"4":5} -> {"1":4,"2":3,"3":0,"4":5}（2人）；{"1":5,"2":3,"3":4,"4":6} -> {"1":5,"2":3,"3":4,"4":6}（2人）；{"1":4,"2":3,"3":2,"4":5} -> {"1":4,"2":3,"3":0,"4":5}（1人） | 请您对这些非西医减重措施的减重效果满意度进行评价。 |
| B15 | B GLP-1药物选择与使用情况 | matrix_numeric | 8 | 0.8 | NA -> {"A":{"interrupted":2,"times":"","avg_interrupt_months":""}}（2人）；{"A":{"interrupted":1,"times":1,"avg_interrupt_months":1}} -> {"A":{"interrupted":2,"times":"","avg_interrupt_months":""}}（2人）；{"A":{"interrupted":2,"times":"","avg_interrupt_months":""}} -> {"A":{"interrupted":1,"times":1,"avg_interrupt_months":1}}（2人） | GLP-1使用过程中，是否有中断一段时间再续用该药物的情况？次数和平均中断时长？ |
| B17 | B GLP-1药物选择与使用情况 | matrix_numeric | 8 | 0.8 | {"3":{"loss_kg":6}} -> {"3":{"loss_kg":7}}（2人）；{"3":{"loss_kg":6}} -> {"3":{"loss_kg":6}}（2人）；NA -> {"3":{"loss_kg":5}}（1人） | 使用各GLP-1药物后，体重下降了多少kg？ |
| B9 | B GLP-1药物选择与使用情况 | matrix | 8 | 0.8 | NA -> {"1":{"used":2,"status":"","start_year_month":""},"2":{"used":2,"status":"","start_year_month":""},"3":{"used":1,"status":1,"start_year_month":"2024-01"},"4":{"used":2,"status":"","start_year_month":""},"5":{"used":2,"status":"","start_year_month":""},"6":{"used":2,"status":"","start_year_month":""}}（1人）；NA -> {"1":{"used":2,"status":"","start_year_month":""},"2":{"used":2,"status":"","start_year_month":""},"3":{"used":1,"status":2,"start_year_month":"2023-03"},"4":{"used":2,"status":"","start_year_month":""},"5":{"used":2,"status":"","start_year_month":""},"6":{"used":2,"status":"","start_year_month":""}}（1人）；{"1":{"used":2,"status":"","start_year_month":""},"2":{"used":2,"status":"","start_year_month":""},"3":{"used":1,"status":1,"start_year_month":"2024-01"},"4":{"used":2,"status":"","start_year_month":""},"5":{"used":2,"status":"","start_year_month":""},"6":{"used":2,"status":"","start_year_month":""}} -> NA（1人） | 请勾选既往使用过的GLP-1药物品牌，并填写当前使用状态和启用时间。 |
| C4 | C 商品名测试 | matrix_rating_1_7 | 8 | 0.8 | 5, 4, 5, 5, 4, 5 -> 6, 5, 5, 6, 4, 5（2人）；6, 5, 5, 6, 4, 5 -> 6, 5, 5, 6, 4, 5（2人）；5, 6, 6, 5, 4, 6 -> 5, 4, 5, 5, 4, 5（2人） | C4 商品名易读性，按商品名1-6顺序导出。 |
| C6 | C 商品名测试 | matrix_rating_1_7 | 8 | 0.8 | {"1":5,"2":5,"3":5,"4":5,"5":4,"6":5} -> {"1":5,"2":6,"3":5,"4":5,"5":4,"6":6}（2人）；{"1":5,"2":5,"3":5,"4":5,"5":4,"6":5} -> {"1":5,"2":5,"3":5,"4":6,"5":4,"6":5}（1人）；{"1":5,"2":5,"3":6,"4":5,"5":5,"6":6} -> {"1":5,"2":6,"3":5,"4":5,"5":4,"6":5}（1人） | 评价各商品名与'重新定义代谢健康，从而改变患者的生活方式、遇见更好的未来'产品理念的相关程度。 |
| C7 | C 商品名测试 | matrix_rating_1_7 | 8 | 0.8 | {"1":5,"2":5,"3":5,"4":5,"5":4,"6":5} -> {"1":5,"2":6,"3":5,"4":5,"5":4,"6":6}（1人）；{"1":5,"2":5,"3":5,"4":5,"5":4,"6":6} -> {"1":6,"2":5,"3":5,"4":6,"5":4,"6":5}（1人）；{"1":5,"2":5,"3":5,"4":6,"5":4,"6":6} -> {"1":6,"2":6,"3":5,"4":6,"5":4,"6":7}（1人） | 评价各商品名与'助力患者迈上通往健康的道路，自由前行'产品理念的相关程度。 |
| A10 | A 减重旅程 | multi | 7 | 0.7 |  -> （1人）； -> [3,4,7]（1人）；NA -> （1人） | 请问您未曾前往医院寻求减重方案的原因有哪些？ |
| A20 | A 减重旅程 | multi | 7 | 0.7 | NA -> NA（2人）；[1,2,3] -> [2,3]（1人）；[1,2,3] -> [2]（1人） | 医生为您处方减重治疗方案中，包含以下哪些治疗方式？ |

## 10. Top变化问题逐人答案对照

| 题目ID | 问卷部分 | pid | 样本编号 | 年龄 | 性别 | BMI | BMI配额段 | 目标人群细分 | 合并症类别汇总 | 无背景答案 | 有背景答案 | 题目文本 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| A6 | A 减重旅程 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 0 | 0, 3, 2, 2, 0, 0, 0, 0, 4, 2, 0 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 1 | 0, 3, 0, 0, 0, 0, 0, 0, 3, 2, 1 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 0, 4, 0, 2, 0, 0, 0, 0, 3, 0, 1 | 0, 3, 0, 2, 0, 0, 0, 0, 3, 0, 1 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | 0, 4, 2, 0, 0, 0, 0, 0, 3, 0, 0 | 0, 3, 0, 2, 0, 0, 0, 0, 3, 0, 0 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 0, 3, 2, 2, 1, 0, 0, 0, 4, 2, 0 | 0, 3, 2, 2, 0, 0, 0, 0, 4, 2, 0 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 0 | 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 1 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | 0, 3, 0, 2, 0, 0, 0, 0, 4, 2, 0 | 0, 3, 2, 0, 0, 0, 0, 0, 5, 0, 0 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | 0, 4, 2, 2, 0, 0, 0, 0, 3, 0, 1 | 0, 3, 0, 2, 0, 0, 0, 0, 3, 0, 2 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 0, 5, 2, 2, 0, 0, 0, 0, 5, 1, 0 | 0, 4, 2, 2, 0, 0, 0, 0, 4, 0, 1 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6 | A 减重旅程 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 0, 5, 2, 2, 0, 0, 0, 0, 3, 0, 0 | 0, 4, 2, 2, 0, 0, 0, 0, 3, 0, 0 | A6 尝试次数，按非西医减重措施1-11顺序导出，未经历该措施填0。 |
| A6a | A 减重旅程 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 0, 6, 0, 3, 0, 0, 0, 0, 4, 2, 0 | 0, 6, 2, 3, 0, 0, 0, 0, 6, 3, 0 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | 0, 6, 0, 3, 0, 0, 0, 0, 4, 2, 3 | 0, 6, 0, 0, 0, 0, 0, 0, 4, 3, 2 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 0, 3, 0, 2, 0, 0, 0, 0, 2, 0, 3 | 0, 4, 0, 2, 0, 0, 0, 0, 3, 0, 2 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | 0, 3, 1, 0, 0, 0, 0, 0, 2, 0, 0 | 0, 3, 0, 2, 0, 0, 0, 0, 2.5, 0, 0 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 0, 6, 2, 3, 2, 0, 0, 0, 6, 4, 0 | 0, 6, 2, 3, 0, 0, 0, 0, 6, 3, 0 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 0, 6, 0, 3, 0, 0, 0, 0, 4, 2, 0 | 0, 6, 0, 3, 0, 0, 0, 0, 4, 2, 6 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | 0, 6, 0, 3, 0, 0, 0, 0, 6, 4, 0 | 0, 6, 1, 0, 0, 0, 0, 0, 4, 0, 0 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | 0, 3, 1, 2, 0, 0, 0, 0, 2, 0, 3 | 0, 6, 0, 3, 0, 0, 0, 0, 4, 0, 3 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6a | A 减重旅程 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 0, 6, 2, 3, 0, 0, 0, 0, 4, 2, 0 | 0, 6, 2, 3, 0, 0, 0, 0, 4, 0, 2 | A6a 单次平均坚持时长，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 0, 300, 0, 200, 0, 0, 0, 0, 0, 200, 0 | 0, 200, 150, 200, 0, 0, 0, 0, 0, 200, 0 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | 0, 300, 0, 200, 0, 0, 0, 0, 0, 200, 600 | 0, 200, 0, 0, 0, 0, 0, 0, 0, 0, 400 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 0, 500, 0, 300, 0, 0, 0, 0, 0, 0, 800 | 0, 200, 0, 300, 0, 0, 0, 0, 0, 0, 400 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | 0, 200, 150, 0, 0, 0, 0, 0, 0, 0, 0 | 0, 200, 0, 300, 0, 0, 0, 0, 0, 0, 0 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | 0, 200, 150, 200, 300, 0, 0, 0, 0, 100, 0 | 0, 200, 150, 200, 0, 0, 0, 0, 0, 200, 0 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | 0, 200, 0, 300, 0, 0, 0, 0, 0, 200, 0 | 0, 200, 0, 300, 0, 0, 0, 0, 0, 200, 400 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | 0, 200, 0, 300, 0, 0, 0, 0, 0, 100, 0 | 0, 200, 100, 0, 0, 0, 0, 0, 0, 0, 0 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | 0, 300, 200, 300, 0, 0, 0, 0, 0, 0, 600 | 0, 300, 0, 400, 0, 0, 0, 0, 0, 0, 600 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 0, 200, 100, 150, 0, 0, 0, 0, 0, 200, 0 | 0, 200, 150, 200, 0, 0, 0, 0, 0, 0, 400 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| A6b | A 减重旅程 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | 0, 300, 200, 300, 0, 0, 0, 0, 0, 0, 0 | 0, 200, 150, 200, 0, 0, 0, 0, 0, 0, 0 | A6b 月均花费，按非西医减重措施1-11顺序导出；对应A6为0时填0。 |
| B13 | B GLP-1药物选择与使用情况 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | NA | {"A":[1,2,6,15]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | {"A":[1,2,4,7,8,15,16]} | {"A":[1,2,4,7,8,15]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | {"A":[1,2,4,7,8,15]} | {"A":[1,2,7,8,15]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | NA | {"A":[1,2,7]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"A":[1,5,7,8,15]} | NA | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | {"A":[1,4,7,15]} | {"A":[1,2,6,15]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | {"A":[1,2,4,7,8,15,16]} | {"A":[1,2,8,15]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":[1,2,6,7,8,15]} | {"A":[1,2,6,7,8]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B13 | B GLP-1药物选择与使用情况 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":[1,2,4,7,8,15]} | {"A":[1,2,6,7,8,15]} | 选择相关GLP-1时，您从哪些信息渠道获取过产品信息？ |
| B14 | B GLP-1药物选择与使用情况 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | NA | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":12,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":20}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":16}} | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":12,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":4}],"total_weeks":24}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":4}],"total_weeks":20}} | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":16,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":24}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | NA | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":20,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":28}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":4}],"total_weeks":20}} | NA | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1mg","maintenance_weeks":16,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":24}} | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":4}],"total_weeks":20}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":4}],"total_weeks":20}} | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":12,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":20}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1mg","maintenance_weeks":12,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":20}} | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.0mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4}],"total_weeks":16}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B14 | B GLP-1药物选择与使用情况 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":8,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":4}],"total_weeks":20}} | {"A":{"start_dose":"0.25mg","start_weeks":4,"maintenance_dose":"1.7mg","maintenance_weeks":20,"transition_doses":[{"dose":"0.5mg","weeks":4},{"dose":"1.0mg","weeks":8}],"total_weeks":36}} | 回忆GLP-1实际使用过程中的剂量调整过程：起始剂量、维持剂量、过渡剂量及各阶段持续周数。 |
| B16 | B GLP-1药物选择与使用情况 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | NA | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重变化的功能"}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重变化的功能"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重变化的功能"}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重变化的功能，方便自己跟踪效果。"}} | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针提醒和记录体重变化的功能"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针提醒和体重记录趋势"}} | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重、腰围变化"}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | NA | {"A":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重、血压等数据的功能"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | NA | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针和复诊提醒、记录体重和血糖变化"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针和复诊提醒、记录体重变化"}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针和复诊的提醒，以及记录体重变化的功能"}} | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针提醒和体重记录曲线，让我更直观看到变化"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"用药提醒和记录体重变化"}} | {"A":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B16 | B GLP-1药物选择与使用情况 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":{"used_tool":1,"satisfaction":6,"most_helpful_feature":"打针提醒和记录体重变化的功能"},"B":{"used_tool":2,"satisfaction":"","most_helpful_feature":""},"C":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | {"A":{"used_tool":2,"satisfaction":"","most_helpful_feature":""}} | 是否借助厂家提供的小程序作为用药记录、提醒和指南支持工具？满意度如何？最有帮助的功能是什么？ |
| B19 | B GLP-1药物选择与使用情况 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | NA | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":5,"12":5,"13":6,"14":5,"15":6,"16":7,"17":4,"18":5,"19":4,"20":3}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | {"A":{"0":6,"1":6,"2":7,"3":5,"4":3,"5":6,"6":7,"7":6,"8":5,"9":6,"10":5,"11":4,"12":4,"13":6,"14":5,"15":6,"16":7,"17":4,"18":6,"19":4,"20":3}} | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":6,"7":5,"8":5,"9":6,"10":5,"11":4,"12":5,"13":6,"14":5,"15":6,"16":7,"17":4,"18":5,"19":4,"20":3}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":4,"12":5,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":5,"12":5,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":5,"7":5,"8":5,"9":6,"10":5,"11":5,"12":5,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | {"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":4,"12":4,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | NA | {"A":{"0":5,"1":5,"2":6,"3":5,"4":3,"5":4,"6":4,"7":4,"8":4,"9":5,"10":5,"11":4,"12":4,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":3,"20":2}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"A":{"0":6,"1":6,"2":7,"3":5,"4":4,"5":6,"6":6,"7":5,"8":5,"9":6,"10":6,"11":4,"12":5,"13":6,"14":5,"15":6,"16":7,"17":4,"18":5,"19":4,"20":5}} | NA | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | {"A":{"0":6,"1":6,"2":7,"3":5,"4":4,"5":6,"6":7,"7":6,"8":5,"9":6,"10":6,"11":5,"12":5,"13":6,"14":5,"15":6,"16":7,"17":4,"18":6,"19":5,"20":4}} | {"A":{"0":6,"1":6,"2":6,"3":5,"4":5,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":5,"12":5,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":4}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | {"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":4,"12":4,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | {"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":4,"7":4,"8":5,"9":5,"10":6,"11":4,"12":4,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":5,"12":5,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | {"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":4,"7":4,"8":4,"9":5,"10":5,"11":4,"12":4,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B19 | B GLP-1药物选择与使用情况 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"A":{"0":6,"1":6,"2":6,"3":5,"4":4,"5":5,"6":5,"7":5,"8":5,"9":6,"10":6,"11":5,"12":5,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":4,"20":3}} | {"A":{"0":5,"1":5,"2":6,"3":4,"4":3,"5":5,"6":5,"7":5,"8":4,"9":5,"10":5,"11":4,"12":4,"13":5,"14":5,"15":6,"16":6,"17":3,"18":4,"19":3,"20":2}} | 结合实际使用体验，对使用过的GLP-1药物在各维度表现的满意度打分。 |
| B3 | B GLP-1药物选择与使用情况 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":7,"12":7,"13":6,"14":5,"15":6,"16":6,"17":4,"18":6,"19":6,"20":5} | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":7,"17":4,"18":5,"19":5,"20":4} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":4,"11":6,"12":6,"13":6,"14":5,"15":5,"16":7,"17":4,"18":6,"19":5,"20":4} | {"1":6,"2":7,"3":5,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":4,"11":5,"12":6,"13":5,"14":5,"15":6,"16":7,"17":4,"18":5,"19":5,"20":6} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":5,"7":6,"8":4,"9":5,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":5,"20":4} | {"1":6,"2":7,"3":5,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":5,"20":4} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 211769103 | 150 | 32 | 男 | 27.13 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 肝病或MASH线索 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":5,"16":6,"17":4,"18":5,"19":5,"20":4} | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 450250102 | 151 | 52 | 男 | 26.35 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5} | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":5,"7":6,"8":4,"9":5,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 411875101 | 187 | 47 | 男 | 26.73 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":6,"16":7,"17":4,"18":5,"19":6,"20":5} | {"1":6,"2":7,"3":5,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 411313102 | 207 | 55 | 男 | 28.55 | 肥胖_BMI>28 | 肥胖且合并糖尿病 | 糖尿病 | {"1":6,"2":7,"3":6,"4":7,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":6,"16":7,"17":4,"18":6,"19":6,"20":5} | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":7,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":5,"20":4} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 220322103 | 47 | 32 | 女 | 28.04 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 关节运动系统 | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":5,"16":6,"17":4,"18":5,"19":5,"20":4} | {"1":6,"2":7,"3":6,"4":7,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":6,"20":5} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 411244104 | 51 | 49 | 女 | 30.85 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"1":6,"2":7,"3":6,"4":7,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":6,"14":5,"15":6,"16":6,"17":4,"18":5,"19":5,"20":4} | {"1":6,"2":7,"3":6,"4":6,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":5,"12":5,"13":5,"14":5,"15":6,"16":6,"17":4,"18":5,"19":5,"20":4} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| B3 | B GLP-1药物选择与使用情况 | 310855102 | 86 | 61 | 女 | 33.3 | 肥胖_BMI>28 | 肥胖未识别糖尿病 | 无可识别研究相关合并症 | {"1":6,"2":7,"3":6,"4":7,"5":5,"6":6,"7":6,"8":5,"9":6,"10":5,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":4,"18":5,"19":5,"20":4} | {"1":6,"2":7,"3":5,"4":6,"5":5,"6":6,"7":6,"8":4,"9":5,"10":4,"11":6,"12":6,"13":5,"14":5,"15":5,"16":6,"17":3,"18":5,"19":6,"20":5} | 选择GLP-1作为减重治疗药物时，以下考虑因素的重要程度分别如何？ |
| C1 | C 新产品测试 | 411879101 | 120 | 55 | 男 | 24.22 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压 | {"X":{"1":4,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":6,"2":5,"3":7,"4":7,"5":4,"6":7},"B":{"1":6,"2":5,"3":5,"4":5,"5":4,"6":6},"C":{"1":6,"2":6,"3":6,"4":6,"5":3,"6":6}} | {"X":{"1":4,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":6,"2":5,"3":7,"4":7,"5":4,"6":6},"B":{"1":6,"2":5,"3":5,"4":5,"5":4,"6":5},"C":{"1":6,"2":5,"3":6,"4":5,"5":3,"6":5}} | 阅读产品X、产品A、产品B、产品C信息后，对各药物在各维度上的表现满意度和总体满意度分别打分。 |
| C1 | C 新产品测试 | 420493101 | 123 | 45 | 男 | 24.51 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 高血压；心脑血管 | {"X":{"1":4,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":7,"2":5,"3":7,"4":7,"5":4,"6":7},"B":{"1":6,"2":5,"3":4,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":5,"4":5,"5":3,"6":5}} | {"X":{"1":5,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":7,"2":5,"3":7,"4":7,"5":4,"6":7},"B":{"1":6,"2":5,"3":5,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":6,"4":6,"5":3,"6":6}} | 阅读产品X、产品A、产品B、产品C信息后，对各药物在各维度上的表现满意度和总体满意度分别打分。 |
| C1 | C 新产品测试 | 673100701 | 138 | 34 | 男 | 24.84 | 超重_24<BMI≤28 | 超重且有可识别合并症 | 关节运动系统 | {"X":{"1":4,"2":5,"3":5,"4":5,"5":3,"6":4},"A":{"1":6,"2":5,"3":7,"4":6,"5":4,"6":6},"B":{"1":6,"2":5,"3":4,"4":5,"5":4,"6":5},"C":{"1":5,"2":5,"3":5,"4":5,"5":3,"6":4}} | {"X":{"1":5,"2":5,"3":5,"4":6,"5":3,"6":5},"A":{"1":6,"2":5,"3":7,"4":7,"5":4,"6":6},"B":{"1":6,"2":5,"3":4,"4":5,"5":4,"6":5},"C":{"1":6,"2":6,"3":5,"4":5,"5":3,"6":5}} | 阅读产品X、产品A、产品B、产品C信息后，对各药物在各维度上的表现满意度和总体满意度分别打分。 |

In [ ]:
from pathlib import Path
import html
import pandas as pd
from IPython.display import HTML, display

RUN_DIR = Path(r'''C:\tmp\obesity_patient_pipeline_branch\obesity_patient_pipeline\data\obesity_questionnaire_10_persona_comparison_md_background''')
pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.max_colwidth', 160)

combined = pd.read_csv(RUN_DIR / 'comparison_answers_combined.csv', dtype=str, keep_default_na=False)
diff = pd.read_csv(RUN_DIR / 'comparison_question_differences.csv', dtype=str, keep_default_na=False)
top_detail = pd.read_csv(RUN_DIR / 'comparison_top10_changed_answer_details.csv', dtype=str, keep_default_na=False)
changed_detail = pd.read_csv(RUN_DIR / 'comparison_changed_answer_details.csv', dtype=str, keep_default_na=False)
persona_change = pd.read_csv(RUN_DIR / 'comparison_persona_change_counts.csv', dtype=str, keep_default_na=False)
demographics = pd.read_csv(RUN_DIR / 'comparison_selected_persona_demographics.csv', dtype=str, keep_default_na=False)
token_summary = pd.read_csv(RUN_DIR / 'comparison_runtime_token_summary.csv', dtype=str, keep_default_na=False)
combined.shape, diff.shape, top_detail.shape


In [ ]:
CJK_FONT_STACK = 'Microsoft YaHei, Microsoft JhengHei, SimHei, SimSun, Noto Sans CJK SC, PingFang SC, Arial, sans-serif'

def barh_svg(df, label_col, value_col, title, width=900):
    chart = df[[label_col, value_col]].copy()
    chart[value_col] = pd.to_numeric(chart[value_col], errors='coerce').fillna(0)
    chart = chart.sort_values(value_col, ascending=True)
    row_h, left, right, top = 30, 145, 40, 52
    height = top + row_h * len(chart) + 32
    plot_w = width - left - right
    max_value = max(float(chart[value_col].max()), 1.0)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             f'<rect width="{width}" height="{height}" fill="white"/>',
             f'<style>text{{font-family:{CJK_FONT_STACK};fill:#202124}} .title{{font-size:18px;font-weight:700}} .label{{font-size:13px}} .value{{font-size:12px;fill:#4b5563}}</style>',
             f'<text class="title" x="{left}" y="28">{html.escape(title)}</text>']
    for i, row in enumerate(chart.to_dict('records')):
        y = top + i * row_h
        value = float(row[value_col])
        bar_w = max(2, value / max_value * plot_w)
        label = html.escape(str(row[label_col]))
        parts += [f'<text class="label" x="{left-10}" y="{y+16}" text-anchor="end">{label}</text>',
                  f'<rect x="{left}" y="{y}" width="{bar_w:.1f}" height="18" rx="3" fill="#2563eb"/>',
                  f'<text class="value" x="{left+bar_w+6:.1f}" y="{y+14}">{value:g}</text>']
    parts.append('</svg>')
    return '\n'.join(parts)

display(HTML(barh_svg(diff.head(20), '题目ID', '答案变化人数', 'Top20题目：答案变化人数')))


In [ ]:
display(HTML(barh_svg(persona_change.sort_values('变化题目数', ascending=False), '样本编号', '变化题目数', '每位persona受背景影响的题目数量')))


In [ ]:
top_questions = diff.head(10)['题目ID'].tolist()
top_detail[top_detail['题目ID'].isin(top_questions)][['题目ID','问卷部分','题目文本','样本编号','年龄','性别','BMI','BMI配额段','目标人群细分','无背景答案','有背景答案']]


In [ ]:
pair_summary = (top_detail.groupby(['题目ID','问卷部分','无背景答案','有背景答案'])
                .size().reset_index(name='人数')
                .sort_values(['题目ID','人数'], ascending=[True, False]))
pair_summary.head(80)


In [ ]:
section_c_questions = ['C1','C2','C3','C4','C5','C6','C7']
section_c_diff = diff[diff['题目ID'].isin(section_c_questions)]
section_c_pairs = (changed_detail[changed_detail['题目ID'].isin(section_c_questions)]
                   .groupby(['题目ID','问卷部分','无背景答案','有背景答案'])
                   .size().reset_index(name='人数')
                   .sort_values(['题目ID','人数'], ascending=[True, False]))
display(section_c_diff[['题目ID','问卷部分','答案变化人数','答案变化率','主要答案迁移','题目文本']])
display(section_c_pairs.head(80))


In [ ]:
section_c_personas = changed_detail[changed_detail['题目ID'].isin(['C1','C2','C3','C4','C5','C6','C7'])]
section_c_personas[['题目ID','样本编号','年龄','性别','BMI','BMI配额段','目标人群细分','合并症类别汇总','无背景答案','有背景答案','题目文本']].head(120)


In [ ]:
persona_profile_cols = ['variant_label','pid','样本编号','目标人群细分','BMI配额段','persona_summary','response_style','assumption_notes']
combined[[c for c in persona_profile_cols if c in combined.columns]].sort_values(['pid','variant_label'])
